# Bag of Words Baseline

### Load the Processed Tables
- The splits, the normalised tags and the shared vocabulary all come from the preparation notebook
- The raw tag column is not read again here

In [1]:
from pathlib import Path

import pandas as pd

candidate_paths = [
    Path("data/processed"),
    Path("../data/processed"),
]
processed_dir = next((path for path in candidate_paths if path.exists()), None)
if processed_dir is None:
    raise FileNotFoundError(
        "Could not find data/processed. Run data_preparation.ipynb first."
    )

splits = pd.read_csv(processed_dir / "splits.csv", dtype={"id": str})
tags_frame = pd.read_csv(processed_dir / "tags_normalised.csv", dtype={"id": str})
vocabulary_frame = pd.read_csv(processed_dir / "shared_vocabulary_50.csv")

tag_vocabulary = list(vocabulary_frame["tag"])

print("Splits shape:", splits.shape)
print("Tags shape:", tags_frame.shape)
print("Tags in the shared vocabulary:", len(tag_vocabulary))
print("Same ids in both tables:", set(splits["id"]) == set(tags_frame["id"]))
splits.head()

Splits shape: (82810, 4)
Tags shape: (82810, 2)
Tags in the shared vocabulary: 50
Same ids in both tables: True


,id,artist,random_split,artist_split
0,0009fFIM1eYThaPg,Cheryl,test,train
1,0010xmHR6UICBOYT,Oddisee,train,train
2,002Jyd0vN4HyCpqL,Blue Öyster Cult,train,test
3,006TYKNjNxWjfKjy,Rhapsody,train,valid
4,007LIJOPQ4Sb98qV,The Chameleons,valid,train


### Read the Lyrics Back
- lyrics_clean.csv stores the counts and the fingerprint, not the text itself
- The same two cleaning functions are reused so the tokens match the preparation step
- The sha1 fingerprints and the word counts are checked against lyrics_clean.csv, which confirms each document sits on the row its id belongs to

In [2]:
import hashlib
import re

candidate_paths = [
    Path("music4all-en"),
    Path("../music4all-en"),
]
data_dir = next((path for path in candidate_paths if path.exists()), None)
if data_dir is None:
    raise FileNotFoundError(
        "Could not find music4all-en. Unzip the subset to the repo root."
    )

lyrics_dir = data_dir / "lyrics"
marker_pattern = re.compile(r"^\s*\[[^\]]{1,30}\]\s*$")


def clean_lines(text):
    kept = []
    for line in text.replace("\r\n", "\n").replace("\r", "\n").split("\n"):
        line = line.rstrip()
        if not line.strip():
            continue
        if marker_pattern.match(line):
            continue
        kept.append(line)

    return "\n".join(kept)


def flatten(text):
    return re.sub(r"\s+", " ", clean_lines(text).lower()).strip()


track_ids = list(splits["id"])
documents = []
for track_id in track_ids:
    path = lyrics_dir / (track_id + ".txt")
    raw = path.read_text(encoding="utf-8", errors="replace")
    documents.append(flatten(raw))

word_counts = pd.Series([len(text.split()) for text in documents], index=track_ids)
fingerprints = pd.Series(
    [hashlib.sha1(text.encode("utf-8")).hexdigest() for text in documents],
    index=track_ids,

)
expected = pd.read_csv(processed_dir / "lyrics_clean.csv", dtype={"id": str}).set_index("id")

print("Documents read:", len(documents))
print("Empty documents:", int((word_counts == 0).sum()))
print("Mean words:", round(word_counts.mean(), 1))
print("Median words:", int(word_counts.median()))
print("Fingerprints matching lyrics_clean.csv:",
      int((fingerprints == expected.loc[track_ids, "fingerprint"]).sum()), "of", len(track_ids))
print("Word counts matching lyrics_clean.csv:",
      int((word_counts == expected.loc[track_ids, "n_words"]).sum()), "of", len(track_ids))

Documents read: 82810
Empty documents: 0
Mean words: 242.9
Median words: 223
Fingerprints matching lyrics_clean.csv: 82810 of 82810
Word counts matching lyrics_clean.csv: 82810 of 82810


### Build the Target Matrix
- One binary column per tag in the shared vocabulary, one row per track, in the order of splits.csv
- Cardinality is the mean number of tags per track and density is cardinality over the vocabulary size

In [3]:
import numpy as np

tag_lists = {}
for record in tags_frame.itertuples(index=False):
    if isinstance(record.tags, str) and record.tags != "":
        tag_lists[record.id] = record.tags.split(",")
    else:
        tag_lists[record.id] = []

position = {}
for i in range(len(tag_vocabulary)):
    position[tag_vocabulary[i]] = i

target = np.zeros((len(track_ids), len(tag_vocabulary)), dtype=np.int8)
row = 0
for track_id in track_ids:
    for tag in tag_lists[track_id]:
        if tag in position:
            target[row, position[tag]] = 1
    row = row + 1

cardinality = target.sum(axis=1).mean()

print("Target shape:", target.shape)
print("Cardinality:", round(float(cardinality), 2))
print("Density:", round(float(cardinality) / len(tag_vocabulary), 4))
print("Rows with no tag:", int((target.sum(axis=1) == 0).sum()))
print("Smallest support over all rows:", int(target.sum(axis=0).min()))
print("Largest support over all rows:", int(target.sum(axis=0).max()))

Target shape: (82810, 50)
Cardinality: 2.46
Density: 0.0493
Rows with no tag: 5092
Smallest support over all rows: 1175
Largest support over all rows: 20727


### Index the Two Splits
- Row positions rather than ids, so the feature matrix and the target can be sliced the same way

In [4]:
split_rows = {}
for name in ["random", "artist"]:
    column = splits[name + "_split"]
    parts = {}
    for part in ["train", "valid", "test"]:
        parts[part] = np.flatnonzero(column == part)
    split_rows[name] = parts

    print(name, "split:",
          "train", len(parts["train"]),
          "valid", len(parts["valid"]),
          "test", len(parts["test"]))

random split: train 57966 valid 12421 test 12423
artist split: train 57997 valid 12424 test 12389


### Bag of Words Features
- Plain word counts with no weighting
- min_df, max_df and the n-gram range are left at their defaults because sweeping them is the next step
- The vectoriser is fitted on the training rows only, so the validation rows contribute no vocabulary

In [5]:
from sklearn.feature_extraction.text import CountVectorizer

documents = np.array(documents, dtype=object)

features = {}
for name in ["random", "artist"]:
    parts = split_rows[name]
    vectoriser = CountVectorizer()
    X_train = vectoriser.fit_transform(documents[parts["train"]])
    X_valid = vectoriser.transform(documents[parts["valid"]])
    features[name] = {
        "vectoriser": vectoriser,
        "X_train": X_train,
        "X_valid": X_valid,
    }

    print(name, "split:",
          "terms", len(vectoriser.vocabulary_),
          "train nonzeros", X_train.nnz,
          "train density", round(X_train.nnz / (X_train.shape[0] * X_train.shape[1]), 6))

random split: terms 86194 train nonzeros 5049464 train density 0.001011


artist split: terms 85307 train nonzeros 5076907 train density 0.001026


### Naive Bayes per Tag
- Binary relevance, so one multinomial Naive Bayes model is fitted for each of the 50 tags
- Nothing is shared across tags except the feature matrix, so this is 50 independent problems

In [6]:
import time

from sklearn.naive_bayes import MultinomialNB

predictions = {}
fit_seconds = {}
for name in ["random", "artist"]:
    parts = split_rows[name]
    X_train = features[name]["X_train"]
    X_valid = features[name]["X_valid"]
    y_train = target[parts["train"]]
    predicted = np.zeros((X_valid.shape[0], len(tag_vocabulary)), dtype=np.int8)
    started = time.perf_counter()
    for j in range(len(tag_vocabulary)):
        model = MultinomialNB()
        model.fit(X_train, y_train[:, j])
        predicted[:, j] = model.predict(X_valid)

    elapsed = time.perf_counter() - started
    predictions[name] = predicted
    fit_seconds[name] = elapsed

    print(name, "split:",
          "models", len(tag_vocabulary),
          "seconds", round(elapsed, 1),
          "seconds per tag", round(elapsed / len(tag_vocabulary), 2))

random split: models 50 seconds 2.0 seconds per tag 0.04


artist split: models 50 seconds 2.0 seconds per tag 0.04


### Micro F1 on Validation
- Micro averaging pools the decisions over every tag and every track before the ratio is taken
- Macro F1 is printed next to it because the two disagree once the tag supports are this uneven
- zero_division=0 keeps a tag that is never predicted at a precision of zero instead of an error

In [7]:
from sklearn.metrics import precision_recall_fscore_support

summary_rows = []
for name in ["random", "artist"]:
    parts = split_rows[name]
    y_valid = target[parts["valid"]]
    predicted = predictions[name]
    micro = precision_recall_fscore_support(
        y_valid, predicted, average="micro", zero_division=0,
    )
    macro = precision_recall_fscore_support(
        y_valid, predicted, average="macro", zero_division=0,
    )
    summary_rows.append({
        "split": name,
        "terms": len(features[name]["vectoriser"].vocabulary_),
        "micro_precision": round(micro[0], 3),
        "micro_recall": round(micro[1], 3),
        "micro_f1": round(micro[2], 3),
        "macro_f1": round(macro[2], 3),
        "true_positive": int(((y_valid == 1) & (predicted == 1)).sum()),
        "false_positive": int(((y_valid == 0) & (predicted == 1)).sum()),
        "false_negative": int(((y_valid == 1) & (predicted == 0)).sum()),
        "fit_seconds": round(fit_seconds[name], 1),
    })

    print(name, "split:",
          "micro precision", round(micro[0], 3),
          "micro recall", round(micro[1], 3),
          "micro F1", round(micro[2], 3),
          "macro F1", round(macro[2], 3))
summary = pd.DataFrame(summary_rows)

display(summary)

random split: micro precision 0.233 micro recall 0.293 micro F1 0.26 macro F1 0.152


artist split: micro precision 0.221 micro recall 0.26 micro F1 0.239 macro F1 0.131


,split,terms,micro_precision,micro_recall,micro_f1,macro_f1,true_positive,false_positive,false_negative,fit_seconds
0,random,86194,0.233,0.293,0.260,0.152,8960,29442,21647,2.0
1,artist,85307,0.221,0.260,0.239,0.131,8026,28216,22856,2.0


### Per Tag Scores
- Precision, recall and F1 for each tag on its own, which is what the macro average is taken over
- predicted is how many validation tracks received the tag, for comparison against support

In [8]:
tag_tables = {}
for name in ["random", "artist"]:
    parts = split_rows[name]
    y_valid = target[parts["valid"]]
    predicted = predictions[name]
    precision, recall, f1, support = precision_recall_fscore_support(
        y_valid, predicted, average=None, zero_division=0,
    )
    tag_tables[name] = pd.DataFrame({
        "tag": tag_vocabulary,
        "support": support,
        "predicted": predicted.sum(axis=0),
        "precision": precision.round(3),
        "recall": recall.round(3),
        "f1": f1.round(3),
    })
table = tag_tables["random"].sort_values("f1", ascending=False)

print("Best five tags on the random split:")
display(table.head(5))
print("Worst five tags on the random split:")
display(table.tail(5))

Best five tags on the random split:


,tag,support,predicted,precision,recall,f1
16,hip-hop,594,894,0.465,0.700,0.559
19,rap,455,763,0.444,0.745,0.557
1,pop,2362,4117,0.368,0.642,0.468
0,rock,3161,4613,0.368,0.537,0.437
38,death metal,273,671,0.265,0.652,0.377


Worst five tags on the random split:


,tag,support,predicted,precision,recall,f1
44,pop rock,200,107,0.019,0.010,0.013
40,dream pop,226,101,0.020,0.009,0.012
46,fip,189,141,0.014,0.011,0.012
48,shoegaze,189,83,0.012,0.005,0.007
45,lo-fi,235,116,0.009,0.004,0.006


### What the Baseline Gets Wrong
- Precision is the weaker of the two numbers, so most of the loss is a tag put on the wrong track
- Only the frequent tags clear an F1 of 0.3 and half the vocabulary sits below 0.1

In [9]:
print("Tags never predicted:", int((table["predicted"] == 0).sum()))
print("Tags predicted more often than their support:",
      int((table["predicted"] > table["support"]).sum()))
print("Tags above F1 0.3:", int((table["f1"] > 0.3).sum()))
print("Tags below F1 0.1:", int((table["f1"] < 0.1).sum()))
print("Correlation between support and F1:",
      round(float(table["support"].corr(table["f1"])), 3))
print("Correlation between support and precision:",
      round(float(table["support"].corr(table["precision"])), 3))

Tags never predicted: 0
Tags predicted more often than their support: 21
Tags above F1 0.3: 7
Tags below F1 0.1: 25
Correlation between support and F1: 0.586
Correlation between support and precision: 0.635


### Compare Against a Trivial Model
- Predicting every tag for every track gives the recall ceiling and the precision floor
- The baseline has to beat this before the score means anything

In [10]:
reference_rows = []
for name in ["random", "artist"]:
    parts = split_rows[name]
    y_valid = target[parts["valid"]]
    always = np.ones(y_valid.shape, dtype=np.int8)
    micro = precision_recall_fscore_support(
        y_valid, always, average="micro", zero_division=0,
    )
    reference_rows.append({
        "split": name,
        "model": "predict every tag",
        "micro_precision": round(micro[0], 3),
        "micro_recall": round(micro[1], 3),
        "micro_f1": round(micro[2], 3),
    })

    print(name, "split, predict every tag:",
          "micro precision", round(micro[0], 3),
          "micro recall", round(micro[1], 3),
          "micro F1", round(micro[2], 3))
display(pd.DataFrame(reference_rows))

random split, predict every tag: micro precision 0.049 micro recall 1.0 micro F1 0.094
artist split, predict every tag: micro precision 0.05 micro recall 1.0 micro F1 0.095


,split,model,micro_precision,micro_recall,micro_f1
0,random,predict every tag,0.049,1.0,0.094
1,artist,predict every tag,0.050,1.0,0.095


### Save

In [11]:
for name in ["random", "artist"]:
    out_path = processed_dir / ("nb_bow_per_tag_" + name + ".csv")
    tag_tables[name].to_csv(out_path, index=False)

    print("Wrote:", out_path)

summary.to_csv(processed_dir / "nb_bow_summary.csv", index=False)

print("Wrote:", processed_dir / "nb_bow_summary.csv")
summary

Wrote: data\processed\nb_bow_per_tag_random.csv


Wrote: data\processed\nb_bow_per_tag_artist.csv
Wrote: data\processed\nb_bow_summary.csv


,split,terms,micro_precision,micro_recall,micro_f1,macro_f1,true_positive,false_positive,false_negative,fit_seconds
0,random,86194,0.233,0.293,0.260,0.152,8960,29442,21647,2.0
1,artist,85307,0.221,0.260,0.239,0.131,8026,28216,22856,2.0
